# Homework 5: Monitoring

This notebook instruments a simple RAG pipeline with OpenTelemetry spans, exports spans to the console and SQLite, and answers the six Homework 5 questions using verified trace data.

## 1. Setup

We import the Week 5 helper modules, use the pinned course commit `8c1834d`, and keep the Qwen/OpenAI-compatible configuration local-only.

In [1]:

import sqlite3
from pathlib import Path
import sys

import pandas as pd

sys.path.insert(0, str(Path.cwd().resolve()))

from monitoring import CollectingSpanExporter, RAGTraced, SQLiteSpanExporter, build_tracer_provider, console_exporter, span_duration_ms
from rag_helper import QUERY, QwenChatAdapter, build_index, load_documents, load_qwen_client


## 2. Starter RAG

We load the 72 lesson pages, build the `minsearch` index, and verify that the required query is available for a simple RAG run.

In [2]:

client, model = load_qwen_client()
documents = load_documents()
assert len(documents) == 72, len(documents)
index = build_index(documents)
rag_base = RAGTraced(index=index, llm_client=QwenChatAdapter(client, model), tracer=build_tracer_provider(CollectingSpanExporter()).get_tracer('bootstrap'))

print('Documents loaded:', len(documents))
print('Query:', QUERY)


Documents loaded: 72
Query: How does the agentic loop keep calling the model until it stops?


## 3. OpenTelemetry Tracing

Each `RAGTraced` method creates its own span: `rag`, `search`, and `llm`. We warm up once to avoid using the first cold call as the reported timing.

In [3]:

warm_provider = build_tracer_provider(CollectingSpanExporter())
warm_tracer = warm_provider.get_tracer('homework_05_monitoring')
warm_rag = RAGTraced(index=index, llm_client=QwenChatAdapter(client, model), tracer=warm_tracer)
warm_result = warm_rag.rag(QUERY)
print('Warmup answer preview:', warm_result['answer'][:220].replace(chr(10), ' '))


Warmup answer preview: The agentic loop keeps calling the model until it stops by using a `while True` loop that checks whether the model’s response contains any function calls. Specifically:  - After each model response is processed, the code


## 4. Console Span Export

For the measured console run, we attach both `ConsoleSpanExporter` and an in-process collector so we can inspect span names, counts, durations, and token attributes.

In [4]:

collector = CollectingSpanExporter()
provider = build_tracer_provider(console_exporter(), collector)
tracer = provider.get_tracer('homework_05_monitoring')
rag = RAGTraced(index=index, llm_client=QwenChatAdapter(client, model), tracer=tracer)
console_result = rag.rag(QUERY)
console_spans = sorted(collector.spans, key=lambda s: s.start_time)

console_span_rows = []
for span in console_spans:
    attrs = span.attributes or {}
    console_span_rows.append({
        'name': span.name,
        'duration_ms': round(span_duration_ms(span), 3),
        'input_tokens': attrs.get('input_tokens'),
        'output_tokens': attrs.get('output_tokens'),
    })

console_span_df = pd.DataFrame(console_span_rows)
console_span_df


{
    "name": "search",
    "context": {
        "trace_id": "0x6eaf2fd066c964fdb5cec5e094efa4af",
        "span_id": "0x39914b66cabfa2eb",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe14deefa9e9929d7",
    "start_time": "2026-07-18T11:45:39.612572Z",
    "end_time": "2026-07-18T11:45:39.612572Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "run_id": "aa7431e8-df6b-413b-9700-225b0a11f609",
        "query_length": 64,
        "num_results": 5,
        "search_result_count": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "8fb1b089-d739-4514-a72b-3479c0de2087",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x6eaf2fd066c964fdb5cec5e094efa4af",
        "span_id": "0x472322f0d9fdf630",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe14deefa9e9929d7",
    "start_time": "2026-07-18T11:45:39.620098Z",
    "end_time": "2026-07-18T11:45:45.340876Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "run_id": "aa7431e8-df6b-413b-9700-225b0a11f609",
        "prompt_length": 30068,
        "provider": "qwen",
        "model": "qwen-plus",
        "input_tokens": 7243,
        "output_tokens": 242,
        "cost": 0.0031876000000000005,
        "duration_ms_local": 5723.213899997063
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "8fb1b089-d739-4514-a72b-34

{
    "name": "rag",
    "context": {
        "trace_id": "0x6eaf2fd066c964fdb5cec5e094efa4af",
        "span_id": "0xe14deefa9e9929d7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-18T11:45:39.612572Z",
    "end_time": "2026-07-18T11:45:45.340876Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "run_id": "aa7431e8-df6b-413b-9700-225b0a11f609",
        "query_length": 64,
        "search_result_count": 5,
        "input_tokens": 7243,
        "output_tokens": 242,
        "cost": 0.0031876000000000005
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "8fb1b089-d739-4514-a72b-3479c0de2087",
            "service.name": "unknown_service"
        },
        "schema_url": 

,name,duration_ms,input_tokens,output_tokens
0,search,0.000,NaN,NaN
1,rag,5728.304,7243.0,242.0
2,llm,5720.778,7243.0,242.0


## 5. Token Attributes

The `llm` span records normalized token attributes using `input_tokens` and `output_tokens`, regardless of whether the underlying client uses Chat Completions usage fields.

In [5]:

llm_console_span = next(row for row in console_span_rows if row['name'] == 'llm')
q1_span_count = len(console_span_rows)
q2_input_tokens = console_result['input_tokens']
q2_output_tokens = console_result['output_tokens']
q3_llm_duration_ms = llm_console_span['duration_ms']

print('Span count:', q1_span_count)
print('Span names:', [row['name'] for row in console_span_rows])
print('LLM input tokens:', q2_input_tokens)
print('LLM output tokens:', q2_output_tokens)
print('Measured LLM duration (ms):', q3_llm_duration_ms)
print('Answer preview:', console_result['answer'][:220].replace(chr(10), ' '))


Span count: 3
Span names: ['search', 'rag', 'llm']
LLM input tokens: 7243
LLM output tokens: 242
Measured LLM duration (ms): 5720.778
Answer preview: The agentic loop keeps calling the model until it stops by using a `while True` loop that checks whether the model's response contains any function calls. Specifically:  - After each model response is processed, the code


## 6. SQLite Span Exporter

We recreate `traces.db`, run the same query four times, and persist every finished span into a local SQLite table called `spans`.

In [6]:

db_path = Path('traces.db')
if db_path.exists():
    db_path.unlink()

sqlite_exporter = SQLiteSpanExporter(db_path)
sqlite_provider = build_tracer_provider(sqlite_exporter)
sqlite_tracer = sqlite_provider.get_tracer('homework_05_monitoring')
sqlite_rag = RAGTraced(index=index, llm_client=QwenChatAdapter(client, model), tracer=sqlite_tracer)

sqlite_runs = []
for _ in range(4):
    result = sqlite_rag.rag(QUERY)
    sqlite_runs.append({
        'run_id': result['run_id'],
        'input_tokens': result['input_tokens'],
        'output_tokens': result['output_tokens'],
        'answer_preview': result['answer'][:180].replace(chr(10), ' '),
    })

sqlite_exporter.shutdown()
pd.DataFrame(sqlite_runs)


,run_id,input_tokens,output_tokens,answer_preview
0,3afc4fd3-a38a-4b10-86ca-7247a0c9c0ee,7243,192,The agentic loop keeps calling the model until...
1,19ca6bb8-dbfa-4683-a8e0-bc17ea84a442,7243,230,The agentic loop keeps calling the model until...
2,63f0509d-fe97-40e3-9485-72571057d1cd,7243,224,The agentic loop keeps calling the model until...
3,dd15b81f-9074-4c33-a687-6869ba87c5b3,7243,205,The agentic loop keeps calling the model until...


## 7. Four-Run Trace Collection

We preview the persisted `spans` table and confirm that the expected span names were written for each run.

In [7]:

with sqlite3.connect(db_path) as conn:
    spans = pd.read_sql_query('SELECT * FROM spans ORDER BY start_time', conn)

assert set(spans['name']) == {'rag', 'search', 'llm'}
spans.head(12)


,trace_id,span_id,parent_id,run_id,name,start_time,end_time,duration_ms,input_tokens,output_tokens,cost
0,c2a1544ed76d8d115bcf274217c92bb6,e3230e8add879554,3b26e2fcb6309a52,3afc4fd3-a38a-4b10-86ca-7247a0c9c0ee,search,1784375145424575900,1784375145425576600,1.0007,NaN,NaN,NaN
1,c2a1544ed76d8d115bcf274217c92bb6,3b26e2fcb6309a52,NaN,3afc4fd3-a38a-4b10-86ca-7247a0c9c0ee,rag,1784375145424575900,1784375149891209900,4466.6340,7243.0,192.0,0.003128
2,c2a1544ed76d8d115bcf274217c92bb6,8ee9448bbf38f229,3b26e2fcb6309a52,3afc4fd3-a38a-4b10-86ca-7247a0c9c0ee,llm,1784375145435453300,1784375149871289500,4435.8362,7243.0,192.0,0.003128
3,e8939393a5c4fbc7ed1b43b159f3da07,fbe357f3255c2a7f,2d8eaebda018cd24,19ca6bb8-dbfa-4683-a8e0-bc17ea84a442,search,1784375149899697400,1784375149901884500,2.1871,NaN,NaN,NaN
4,e8939393a5c4fbc7ed1b43b159f3da07,2d8eaebda018cd24,NaN,19ca6bb8-dbfa-4683-a8e0-bc17ea84a442,rag,1784375149899697400,1784375155142062700,5242.3653,7243.0,230.0,0.003173
5,e8939393a5c4fbc7ed1b43b159f3da07,2b23f16a3ef2fa60,2d8eaebda018cd24,19ca6bb8-dbfa-4683-a8e0-bc17ea84a442,llm,1784375149909494300,1784375155122583000,5213.0887,7243.0,230.0,0.003173
6,214549123e4e7e5a36ba2f8e00109aec,a327fa447cf1df57,72e081e6a59a2454,63f0509d-fe97-40e3-9485-72571057d1cd,search,1784375155152335400,1784375155154462900,2.1275,NaN,NaN,NaN
7,214549123e4e7e5a36ba2f8e00109aec,72e081e6a59a2454,NaN,63f0509d-fe97-40e3-9485-72571057d1cd,rag,1784375155152335400,1784375160214533600,5062.1982,7243.0,224.0,0.003166
8,214549123e4e7e5a36ba2f8e00109aec,b06cb4b7abcc41f7,72e081e6a59a2454,63f0509d-fe97-40e3-9485-72571057d1cd,llm,1784375155162976800,1784375160206051600,5043.0748,7243.0,224.0,0.003166
9,f7d13e22da7eeac51d23de0d117bde58,8893ae99bc4f61d4,cb2ff8853a8a4637,dd15b81f-9074-4c33-a687-6869ba87c5b3,search,1784375160224162700,1784375160225738700,1.5760,NaN,NaN,NaN


## 8. SQLite Analysis

From the persisted spans we compute duration totals by span name, inspect the four `llm` input token values, and derive the final Homework 5 answers.

In [8]:

span_names = sorted(spans['name'].unique().tolist())
duration_totals = spans[spans['name'] != 'rag'].groupby('name', as_index=False)['duration_ms'].sum().sort_values('duration_ms', ascending=False)
llm_spans = spans[spans['name'] == 'llm'].copy()
llm_input_tokens = llm_spans['input_tokens'].astype(int).tolist()
llm_output_tokens = llm_spans['output_tokens'].astype(int).tolist()
llm_durations = llm_spans['duration_ms'].round(3).tolist()

min_input_tokens = min(llm_input_tokens)
max_input_tokens = max(llm_input_tokens)
absolute_difference = max_input_tokens - min_input_tokens
percentage_difference_min = 0.0 if min_input_tokens == 0 else absolute_difference / min_input_tokens * 100
percentage_difference_mean = absolute_difference / (sum(llm_input_tokens) / len(llm_input_tokens)) * 100

duration_totals


,name,duration_ms
0,llm,19793.2383
1,search,6.8913


In [9]:

print('Span names:', span_names)
print('LLM durations (ms):', llm_durations)
print('LLM input tokens across four runs:', llm_input_tokens)
print('LLM output tokens across four runs:', llm_output_tokens)
print('Min input tokens:', min_input_tokens)
print('Max input tokens:', max_input_tokens)
print('Absolute difference:', absolute_difference)
print('Percentage difference vs min:', percentage_difference_min)
print('Percentage difference vs mean:', percentage_difference_mean)


Span names: ['llm', 'rag', 'search']
LLM durations (ms): [4435.836, 5213.089, 5043.075, 5101.239]
LLM input tokens across four runs: [7243, 7243, 7243, 7243]
LLM output tokens across four runs: [192, 230, 224, 205]
Min input tokens: 7243
Max input tokens: 7243
Absolute difference: 0
Percentage difference vs min: 0.0
Percentage difference vs mean: 0.0


## 9. Final Answers

This summary maps the measured values to the official multiple-choice options.

In [10]:

q1_option = 3
q2_option = 7000
q3_option = 'Over 2000ms'
q4_option = 'rag, search, and llm'
q5_option = duration_totals.iloc[0]['name']
q6_option = 'They are identical' if absolute_difference == 0 else 'Within 10% of each other'

final_answers = {
    'Q1_span_count': q1_span_count,
    'Q1_option': q1_option,
    'Q2_input_tokens': q2_input_tokens,
    'Q2_output_tokens': q2_output_tokens,
    'Q2_option': q2_option,
    'Q3_llm_duration_ms': q3_llm_duration_ms,
    'Q3_option': q3_option,
    'Q4_span_names': span_names,
    'Q4_option': q4_option,
    'Q5_duration_totals_excluding_rag': duration_totals.to_dict(orient='records'),
    'Q5_option': q5_option,
    'Q6_input_tokens': llm_input_tokens,
    'Q6_absolute_difference': absolute_difference,
    'Q6_percentage_difference_min': percentage_difference_min,
    'Q6_percentage_difference_mean': percentage_difference_mean,
    'Q6_option': q6_option,
}

final_answers


{'Q1_span_count': 3,
 'Q1_option': 3,
 'Q2_input_tokens': 7243,
 'Q2_output_tokens': 242,
 'Q2_option': 7000,
 'Q3_llm_duration_ms': 5720.778,
 'Q3_option': 'Over 2000ms',
 'Q4_span_names': ['llm', 'rag', 'search'],
 'Q4_option': 'rag, search, and llm',
 'Q5_duration_totals_excluding_rag': [{'name': 'llm',
   'duration_ms': 19793.2383},
  {'name': 'search', 'duration_ms': 6.8913}],
 'Q5_option': 'llm',
 'Q6_input_tokens': [7243, 7243, 7243, 7243],
 'Q6_absolute_difference': 0,
 'Q6_percentage_difference_min': 0.0,
 'Q6_percentage_difference_mean': 0.0,
 'Q6_option': 'They are identical'}